In [ ]:
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# --- 1. FILE & METADATA CONFIGURATION ---
raw_file_path = "/Users/wigabaihaqi/Library/CloudStorage/GoogleDrive-wiga@amikompurwokerto.ac.id/My Drive/Doktor Teknik Elektro/Semester II/Eksperimen/Vektor PLV/dataset/4. visual/Subject 10/subject10.csv"
learning_style = "visual"
subject_index = 10
subject_name = "subject10"
start_time_str = "30-11-2024 16:33:58.981"

# --- 2. LOAD DATA & PREPROCESSING ---
df_raw = pd.read_csv(raw_file_path, header=0)
df_raw = df_raw.rename(columns=lambda x: x.strip() if isinstance(x, str) else x)
df_raw["Timestamp (Formatted)"] = pd.to_datetime(df_raw["Timestamp (Formatted)"])

# Butterworth Band-pass Filter (8-30 Hz)
lowcut, highcut, fs, order = 8.0, 30.0, 250.0, 4
nyquist = 0.5 * fs
b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")

df_filtered = df_raw.copy()
channels_list = [
    "EXG Channel 0",
    "EXG Channel 1",
    "EXG Channel 2",
    "EXG Channel 3",
    "EXG Channel 4",
    "EXG Channel 5",
    "EXG Channel 6",
    "EXG Channel 7",
]
for channel in channels_list:
    df_filtered[channel] = filtfilt(b, a, df_raw[channel])

column_mapping = {
    "EXG Channel 0": "0",
    "EXG Channel 1": "1",
    "EXG Channel 2": "2",
    "EXG Channel 3": "3",
    "EXG Channel 4": "4",
    "EXG Channel 5": "5",
    "EXG Channel 6": "6",
    "EXG Channel 7": "7",
}
df_mapped = df_filtered.rename(columns=column_mapping)
selected_channels = ["0", "1", "2", "3", "4", "5", "6", "7"]


def calculate_end_time(start_time, duration):
    start_datetime = pd.to_datetime(start_time, format="%d-%m-%Y %H:%M:%S.%f")
    end_datetime = start_datetime + timedelta(seconds=duration)
    return end_datetime


# --- 3. 1-SECOND WINDOW EXTRACTION ---
filename = f"{learning_style}-{subject_name}-preprocessed.csv"
output_folder = "2. Preprocessed-verbal-visual"
os.makedirs(output_folder, exist_ok=True)
output_file_path = os.path.join(output_folder, filename)

time_window_duration = 1  # 1 second per window

for i in range(1, 2):
    session_stimulant = i
    if session_stimulant == 1:
        trial_id = 1
        n_iterations = 10
        current_start_time = start_time_str
        with open(output_file_path, "w") as f:
            f.truncate(0)

    print(f"Session #{session_stimulant}")

    for iteration in range(1, n_iterations + 1):
        question_id = n_iterations * (session_stimulant - 1) + iteration
        print(f"Stimulant #{question_id}:")

        for session in range(1, 3):  # 2 sub-sessions per iteration (question & answer)

            for window in range(1, 16):  # 15 time-windows per session
                end_time = calculate_end_time(current_start_time, time_window_duration)

                # Slice EEG data for the selected 1-second window
                selected_data = df_mapped[
                    (df_mapped["Timestamp (Formatted)"] >= current_start_time)
                    & (df_mapped["Timestamp (Formatted)"] <= end_time)
                ].reset_index(drop=True)

                # Process ALL CHANNELS synchronously for the same window
                for channel in selected_channels:
                    datapoints = selected_data[channel][:250].tolist()
                    original_length = len(datapoints)

                    # Interpolate to 250 points if samples are less than 250
                    if original_length < 250:
                        x = np.arange(original_length)
                        y = np.array(datapoints)
                        x_new = np.linspace(0, original_length - 1, 250)
                        datapoints = np.interp(x_new, x, y).tolist()

                    new_row_data = {
                        "trial": trial_id,
                        "subject": subject_name,
                        "timestamp_start": current_start_time,
                        "timestamp_end": end_time,
                        "stimulant": question_id,
                        "session": session,
                        "window": window,
                        "channel": channel,
                        "nomor": ((subject_index - 1) * (n_iterations * 2))
                        + (question_id - 1) * 2
                        + session,
                        **{
                            f"datapoint{idx+1}": val
                            for idx, val in enumerate(datapoints)
                        },
                        "class": learning_style,
                    }

                    new_row_df = pd.DataFrame(new_row_data, index=[0])
                    with open(output_file_path, "a") as f:
                        new_row_df.to_csv(f, header=f.tell() == 0, index=False)

                    trial_id += 1

                # MAIN FIX: Update current_start_time AFTER all 8 channels are processed for this window
                current_start_time = pd.to_datetime(
                    end_time, format="%d-%m-%Y %H:%M:%S.%f"
                ) + pd.to_timedelta(1, unit="ms")
                current_start_time = current_start_time.strftime("%d-%m-%Y %H:%M:%S.%f")[:-3]

print("done!")

Session #1
Stimulant #1:
Stimulant #2:
Stimulant #3:
Stimulant #4:
Stimulant #5:
Stimulant #6:
Stimulant #7:
Stimulant #8:
Stimulant #9:
Stimulant #10:
done!
